[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-4-llms-genai/10-fine-tuning-qlora-and-alignment/code/fine_tuning_handson.ipynb)

# Class 4.10: Fine-tuning II, hands-on QLoRA and alignment

Run the QLoRA fine-tune prepared in class 4.9, evaluate the tuned model against the
base honestly, then merge and serve it. Alignment (how models are shaped after
pretraining) is covered in the slides; the milestone note at the end ties it together.

**What we will cover:** launch the QLoRA run (the class 2.2 loop on the LoRA add-on),
load a pre-baked checkpoint so we do not wait, compare tuned vs base on house-style
adherence and the class 4.8 eval questions, then merge the adapter into one servable
model. Run on a free-tier Colab **T4 GPU (16 GB)** runtime.

## Setup

Same fine-tuning stack as class 4.9. `bitsandbytes` needs a GPU.

Install the necessary libraries in your virtual environment:

`pip install transformers datasets peft trl bitsandbytes accelerate`

## 1. Load the dataset and split (from class 4.9)

The house-style dataset and the train/validation split are the same as 4.9; the
class 4.8 eval questions are our held-out test, never trained on.

In [1]:
import json
from datasets import Dataset

rows = [json.loads(l) for l in open("data/dataset.jsonl")]
ds = Dataset.from_list(rows)
split = ds.train_test_split(test_size=0.2, seed=42)
train_ds, val_ds = split["train"], split["test"]

eval_questions = [json.loads(l)["question"] for l in open("data/eval_set.jsonl")]
print("train:", len(train_ds), "| val:", len(val_ds), "| held-out eval:", len(eval_questions))

c:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-ai-july-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train: 57 | val: 15 | held-out eval: 30


## 2. Configs: 4-bit base, LoRA add-on, trainer (from class 4.9)

In [2]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)

# 4-bit base (the "Q" in QLoRA): store the frozen model in 4-bit to fit a free GPU.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,                      # store base weights in 4-bit (~4x smaller)
    bnb_4bit_quant_type="nf4",              # NormalFloat4: 4-bit format tuned to the weight distribution
    bnb_4bit_use_double_quant=True,         # quantize the quant constants too; a little more saved
    bnb_4bit_compute_dtype=torch.bfloat16,  # dequantize to bfloat16 for the matmuls so the math stays stable
)
# LoRA add-on: freeze the base, train a small low-rank update W' = W + BA.
lora = LoraConfig(
    r=4,                # rank of B,A (main capacity/size knob)
    lora_alpha=8,       # scaling; update applied as (alpha/r)*BA = 2x here
    lora_dropout=0.05,   # light regularization on the adapter
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # attention q/k/v/output projections get adapters
    task_type="CAUSAL_LM",  # next-token language model
)
# Trainer knobs. output_dir="dka-lora": "dka" = Domain Knowledge Assistant (the
# Module 4 milestone); "lora" = a LoRA adapter (merged into "dka-merged" below).
cfg = SFTConfig(
    output_dir="dka-lora",
    num_train_epochs=3,             # passes over the data
    per_device_train_batch_size=2,  # examples per GPU step (small for VRAM)
    gradient_accumulation_steps=4,  # effective batch 2*4 = 8 without the memory for 8
    learning_rate=2e-4,             # higher LR is fine since only the adapter trains
    logging_steps=5,                # print loss every 5 steps
    save_strategy="epoch",          # checkpoint each epoch
)

W0923 21:20:22.660000 30612 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


### How the 4-bit base and 16-bit adapter work together

The `bnb` config above stores the base in 4-bit, but the math still runs in 16-bit.
Here is the trick on one neuron with four weights.

- **True weights (bf16):** `[ 0.42, -0.13, 0.05, -0.37 ]` (16 bits each).
- **Stored in VRAM (4-bit):** codes `[ 15, 6, 8, 4 ]` (4 bits each) plus one shared
  `scale = 0.42` per block. This is all that sits in memory (the ~4x shrink).
- **Rebuilt for the matmul (bf16):** `table[code] * scale = [ 0.42, -0.13, 0.01, -0.37 ]`,
  used once in `W . x`, then **discarded**, never written back. Next time it is rebuilt
  from the same codes.

Dequantization is lossy (`0.05` came back as `0.01`), so the base is permanently a
little wrong. That is fine: the frozen base only needs to be roughly right, and the
LoRA add-on (`B, A`), which is what actually learns, stays full 16-bit and is never
quantized. Mental model: **4-bit is how the weight is filed away; 16-bit is the
photocopy the GPU makes for one calculation, then shreds.**

## 3. Kick off the QLoRA run

This is the real training run: the class 2.2 loop (forward, loss, backprop, update)
on the LoRA add-on over the frozen 4-bit base. It is slow, so once you have seen the
loss start to fall, stop the cell and continue with a pre-baked checkpoint below.

In [3]:
from transformers import AutoModelForCausalLM
from trl import SFTTrainer

# quantization_config=bnb loads it in 4-bit; device_map="auto" places layers on the GPU(s) automatically
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
# passing peft_config=lora is what makes this a LoRA fine-tune rather than a full one
trainer = SFTTrainer(model=base, args=cfg, train_dataset=train_ds,
                     eval_dataset=val_ds, peft_config=lora)
trainer.train()   # slow: interrupt after a few logging steps, then use the pre-baked checkpoint

Truncating train dataset: 100%|██████████| 57/57 [00:00<00:00, 7000.95 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 57/57 [00:00<00:00, 18896.25 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 15/15 [00:00<00:00, 5696.20 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
5,3.871099
10,2.964172
15,2.687067
20,2.445990


TrainOutput(global_step=24, training_loss=2.890588084856669, metrics={'train_runtime': 40.8795, 'train_samples_per_second': 4.183, 'train_steps_per_second': 0.587, 'total_flos': 45180491845632.0, 'train_loss': 2.890588084856669, 'entropy': 2.476982061679547, 'num_tokens': 20109.0, 'mean_token_accuracy': 0.48448702005239636, 'epoch': 3.0})

## 4. Load the pre-baked checkpoint (the pre-bake pattern)

Rather than wait for the full run, load an adapter that was trained ahead of time.
Set `ADAPTER` to the instructor's pre-baked adapter, or to your own `dka-lora` output
once your run finishes.

In [4]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

ADAPTER = "dka-lora/checkpoint-24"   # the "dka-lora" adapter dir from class 4.9 (Domain Knowledge Assistant + LoRA); pre-baked, or your finished run's output_dir
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
tuned = PeftModel.from_pretrained(base_model, ADAPTER)  # load the adapter on top of the frozen base
print("loaded tuned model from", ADAPTER)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 432.45it/s]


loaded tuned model from dka-lora/checkpoint-24


## 5. Evaluate tuned vs base: house-style adherence

Did the fine-tune teach the behavior? We score a simple, deterministic house-style
metric on the held-out eval questions: does the answer cite a publication and year,
and stay concise? Compare base vs tuned. (You can also run the full class 4.8 metrics
here for faithfulness and relevance.)

In [5]:
import re

def generate(model, question, max_new_tokens=80):   # max_new_tokens caps answer length (answers are meant to be concise)
    msgs = [{"role": "user", "content": question}]
    # add_generation_prompt=True appends the assistant-turn marker so the model continues as the assistant
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)   # return_tensors="pt": PyTorch tensors; move to the model's device
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)  # do_sample=False = greedy/deterministic, so base-vs-tuned is a fair comparison
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # slice off the prompt tokens; drop role markers

def house_style(answer):
    cites = bool(re.search(r"\(Pub \d+.*20\d\d\)", answer))   # cites (Pub NNN, YYYY)
    concise = answer.count(".") <= 2                              # at most ~2 sentences
    return float(cites and concise)

def score(model):
    return sum(house_style(generate(model, q)) for q in eval_questions) / len(eval_questions)

print("base  house-style adherence:", round(score(base_model), 2))
print("tuned house-style adherence:", round(score(tuned), 2))
# Expected: tuned scores clearly higher (it learned to cite and stay concise). Also
# spot-check that general answers did not get worse (guard against overfitting).

base  house-style adherence: 0.0
tuned house-style adherence: 0.0


## 6. Merge and serve

Fold the LoRA add-on back into the base (`W' = W + BA`) to get one standalone model,
then save it. Serve it like any model, through an endpoint, or convert to GGUF for Ollama.

**On precision:** the LoRA add-on is 16-bit and never quantized. If you keep it
separate, the base stays 4-bit and does the same dequantize-then-matmul at inference,
with the adapter's small 16-bit term on top. **Merging must dequantize the 4-bit base**
to add `BA`, so `dka-merged` is a full **16-bit** model, not 4-bit. To ship a small
artifact, re-quantize it afterward (for example export to 4-bit GGUF for Ollama). 

**Some articles to follow**
- [Convert a Fine-Tuned Model to GGUF for Ollama](https://llmconfigurator.com/en/guides/convert-model-to-gguf)
- [The easiest way to convert a model to GGUF and Quantize](https://medium.com/@qdrddr/the-easiest-way-to-convert-a-model-to-gguf-and-quantize-91016e97c987)
- [Tutorial: How to convert HuggingFace model to GGUF format](https://github.com/ggml-org/llama.cpp/discussions/2948)

In [8]:
merged = tuned.merge_and_unload()      # bake the adapter into the weights
merged.save_pretrained("dka-merged")
tok.save_pretrained("dka-merged")
print("merged model saved to dka-merged/  (serve via an endpoint, or convert to GGUF for Ollama)")

c:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-ai-july-2026\.venv\Lib\site-packages\peft\tuners\lora\bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.09it/s]

merged model saved to dka-merged/  (serve via an endpoint, or convert to GGUF for Ollama)


## 7. Alignment (recap of the slides)

Pretraining gives raw next-token skill; **alignment** shapes the model to follow
instructions and be helpful, in stages: SFT (what you just ran), then **preference
optimization**, learning from better-vs-worse answers.

- **DPO** (Direct Preference Optimization) is the practical default. Preferences are
  collected *offline* into a fixed dataset of (prompt, chosen, rejected), then trained
  in one pass against a frozen reference model, no reward model, no RL loop. The labels
  come from human raters or from an AI judge, which is **RLAIF** (RL from AI feedback;
  Anthropic's Constitutional AI is the known example). The DPO pass itself can use LoRA.
- **PPO with RLHF** (RL from human feedback) is the ancestor DPO simplified: it trains
  a separate reward model, then an RL loop, powerful but complex.
- **RLVR** (RL from verifiable rewards) is a *reward source*: it needs prompts with
  ground truth (a math answer key, a code test suite), and a deterministic verifier
  returns pass/fail instead of a human or reward model.
- **GRPO** (Group Relative Policy Optimization) is an *algorithm*: a PPO variant with no
  value network that samples a group of answers per prompt and pushes toward those above
  the group average. GRPO is **not** a subset of RLVR (reward source vs optimizer); they
  are usually paired to train today's reasoning models.

Alignment is not just refusals; it is what makes a model usable at all.

## Milestone, part 2

Add this fine-tuned component to your Domain Knowledge Assistant, and write a short,
honest note titled "should I have used RAG instead?", arguing why fine-tuning was, or
was not, the right lever for this behavior. Together with the RAG app (part 1), this
completes the Module 4 milestone. In Module 5 the assistant becomes an agent's first tool.